# モデルのチューニング

In [7]:
# ライブラリのインポート
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
from datetime import datetime

from tqdm import tqdm
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, roc_auc_score
from sklearn.inspection import permutation_importance
import xgboost as xgb
import shap
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.figsize'] = (12, 8)

In [2]:
x_train = pd.read_csv('../data/treated-data/df_x_train.csv')
y_train = pd.read_csv('../data/treated-data/df_y_train.csv')
x_test = pd.read_csv('../data/treated-data/df_x_test.csv')
y_test = pd.read_csv('../data/treated-data/df_y_test.csv')

In [10]:
# スコアの設定
ms = 0

myparams = {xgb.XGBClassifier(use_label_encoder=False, eval_metric="logloss"): {
    'max_depth': [3, 4, 5, 6],
    'eta': [0.01, 0.05, 0.1, 0.2],
    'objective': ['multi:softmax'],
    'num_class': [2],
    'n_estimators': [i for i in range(10, 101, 10)]
}}

# グリッドサーチによる探索
for model, params in myparams.items():
    # パラメータを振って当てはめる
    xgb_clf = GridSearchCV(model, params)
    xgb_clf.fit(x_train, y_train.values.ravel() if isinstance(y_train, pd.DataFrame) else y_train)
    # 予測と予測精度確認
    pred = xgb_clf.predict(x_test)
    sc = f1_score(y_test, pred, average="micro")
    # これまでのスコアより新しいのが優れていたら更新する
    if ms < sc:
        ms = sc
        best_param = xgb_clf.best_params_
        best_model = model.__class__.__name__

print(ms)
print(best_model)
print(best_param)

0.9531938325991189
XGBClassifier
{'eta': 0.2, 'max_depth': 6, 'n_estimators': 100, 'num_class': 2, 'objective': 'multi:softmax'}


In [13]:
# XGBoostのモデルをPickleで保存
datetime_str = datetime.now().strftime('%Y%m%d_%H%M%S')
model_filename = f'../models/xgb_gbm_{datetime_str}.pkl'
with open(model_filename, 'wb') as f:
    pickle.dump(best_model, f)
    print(f"モデルを {model_filename} に保存しました。")

モデルを ../models/xgb_gbm_20251201_162903.pkl に保存しました。
